# Advanced Analytics: Product Performance Report (`gold.report_products`)

## Executive Summary & Architecture
This notebook builds and validates the **Product Performance Analytical Data Mart** (`gold.report_products`). Consolidating product catalog metadata, sales volume, lifetime revenue, pricing health, and recency into a unified analytical view empowers:
- Merchandising and inventory managers with SKU-level performance metrics.
- Pricing analysts with realized average selling prices vs manufacturer cost.
- Executive leadership with automated product segment tiering.

## Schema Specification (16 Columns)
1. `product_key`: Surrogate key from `gold.dim_products`.
2. `product_name`: Descriptive name of the item.
3. `category`: Top-level taxonomy category.
4. `subcategory`: Detailed sub-tier category classification.
5. `cost`: Unit manufacturing cost from `gold.dim_products`.
6. `last_sale_date`: Most recent order date for the item.
7. `recency_in_months`: Months elapsed since the product was last sold.
8. `product_segment`: Performance classification (`'High-Performer'`, `'Mid-Range'`, `'Low-Performer'`).
9. `lifespan`: Duration in months between first and last sale.
10. `total_orders`: Count of distinct orders containing the product.
11. `total_sales`: Aggregate revenue generated.
12. `total_quantity_sold`: Total physical units sold.
13. `total_customers`: Count of unique customers purchasing the product.
14. `avg_selling_price`: Realized average price per unit (`sales_amount / quantity`).
15. `avg_order_revenue`: Average revenue generated per order (AOR = `total_sales / total_orders`).
16. `avg_monthly_revenue`: Mean revenue generated per active month (`total_sales / lifespan`).

## Dialect Translation & Safety Rules
- **Recency Calculation**: Computes customer recency metrics using Spark SQL `months_between()`:
  - Lifespan: `cast(floor(months_between(max(order_date), min(order_date))) as int)`
  - Recency: `cast(floor(months_between(current_date(), last_sale_date)) as int)`
- **Division-by-Zero Protection**:
  - `avg_selling_price`: `round(avg(cast(sales_amount as double) / nullif(quantity, 0)), 1)`
  - `avg_order_revenue`: `case when total_orders = 0 then 0.0 else cast(total_sales as double) / total_orders end`
  - `avg_monthly_revenue`: `case when lifespan = 0 then cast(total_sales as double) else round(cast(total_sales as double) / lifespan, 0) end`
- **Column Parity**: Selects `cost` on `gold.dim_products`.

## Data Contracts & Layer Dependencies
- **Target View Created**: `gold.report_products`
- **Source Tables Referenced**:
  - `gold.fact_sales` (`order_number`, `order_date`, `customer_key`, `sales_amount`, `quantity`, `product_key`)
  - `gold.dim_products` (`product_key`, `product_name`, `category`, `subcategory`, `cost`)

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Setup display helper and session verification
try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure gold database context
spark.sql("USE gold")
print("Session configured. Active database context: gold")

## 1. DDL & Mart Transformation: Create `gold.report_products`

### Transformation Pipeline
1. `base_query`: Joins sales facts to product dimensions, ensuring valid sales transactions.
2. `product_aggregations`: Aggregates orders, sales volume, distinct buyers, active lifespan, and realized unit price.
3. `final_projection`: Appends recency in months, commercial product segments, AOR, and monthly revenue run rates.

In [ ]:
# COMMAND ----------
# 1. Create or Replace View: gold.report_products
spark.sql("""
CREATE OR REPLACE VIEW gold.report_products AS
WITH base_query AS (
    SELECT 
        f.order_number,
        f.order_date,
        f.customer_key,
        f.sales_amount,
        f.quantity,
        p.product_key,
        p.product_name,
        p.category,
        p.subcategory,
        p.cost
    FROM gold.fact_sales f
    LEFT JOIN gold.dim_products p
        ON f.product_key = p.product_key
    WHERE f.order_date IS NOT NULL
),
product_aggregations AS (
    SELECT
        product_key,
        product_name,
        category,
        subcategory,
        cost,
        max(order_date) AS last_sale_date,
        cast(floor(months_between(max(order_date), min(order_date))) as int) AS lifespan,
        count(distinct order_number) AS total_orders,
        count(distinct customer_key) AS total_customers,
        sum(sales_amount) AS total_sales,
        sum(quantity) AS total_quantity_sold,
        round(avg(cast(sales_amount as double) / nullif(quantity, 0)), 1) AS avg_selling_price 
    FROM base_query
    GROUP BY
        product_key,
        product_name,
        category,
        subcategory,
        cost
)
SELECT 
    product_key,
    product_name,
    category,
    subcategory,
    cost,
    last_sale_date,
    cast(floor(months_between(current_date(), last_sale_date)) as int) AS recency_in_months,
    CASE
        WHEN total_sales > 50000 THEN 'High-Performer'
        WHEN total_sales >= 10000 THEN 'Mid-Range'
        ELSE 'Low-Performer'
    END AS product_segment,
    lifespan,
    total_orders,
    total_sales,
    total_quantity_sold,
    total_customers,
    avg_selling_price,
    CASE
        WHEN total_orders = 0 THEN 0.0
        ELSE cast(total_sales as double) / total_orders 
    END AS avg_order_revenue,
    CASE
        WHEN lifespan = 0 THEN cast(total_sales as double)
        ELSE round(cast(total_sales as double) / lifespan, 0)
    END AS avg_monthly_revenue
FROM product_aggregations
""")
print("View 'gold.report_products' created and registered successfully.")

## 2. Product Segment Distribution & Commercial Health

### Business Context & Objective
Aggregates product performance across tiers (High-Performer, Mid-Range, Low-Performer) to measure revenue concentration across the catalog.

In [ ]:
# COMMAND ----------
# 2. Product Segment Breakdown
df_product_segment_summary = spark.sql("""
SELECT
    product_segment,
    count(product_key) AS total_products,
    sum(total_sales) AS segment_revenue,
    round(avg(total_sales), 2) AS avg_product_revenue,
    round(avg(avg_selling_price), 2) AS avg_realized_price,
    round(avg(recency_in_months), 1) AS avg_recency_months
FROM gold.report_products
GROUP BY product_segment
ORDER BY segment_revenue DESC
""")
display(df_product_segment_summary)

## 3. Top 10 Product Performers Inspection

### Business Context & Objective
Inspects the top 10 revenue-generating products in the enterprise catalog.

In [ ]:
# COMMAND ----------
# 3. Top 10 Revenue-Generating Products
df_top_performers = spark.sql("""
SELECT
    product_name,
    category,
    subcategory,
    cost,
    avg_selling_price,
    product_segment,
    recency_in_months,
    total_orders,
    total_sales,
    total_quantity_sold,
    round(avg_order_revenue, 2) AS avg_order_revenue
FROM gold.report_products
ORDER BY total_sales DESC
LIMIT 10
""")
display(df_top_performers)

### Strategic Merchandising Recommendations
1. **Catalog Rationalization**: Review Low-Performer items with high recency (> 12 months since last sale) for catalog retirement or clearance.
2. **Gross Margin Protection**: Audit items where `avg_selling_price` approaches or falls below `cost` due to excessive promotional discounting.
3. **High-Performer Cross-Selling**: Feature top-performing flagship products prominently on e-commerce landing pages.